<a href="https://colab.research.google.com/github/kho126942-a11y/makemore/blob/master/torch_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt
# makemore에서 names.txt파일에 들어가 raw를 누르고 링크를 얻는다.

--2026-04-15 04:06:38--  https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.01s   

2026-04-15 04:06:38 (17.9 MB/s) - ‘names.txt’ saved [228145/228145]



In [ ]:
words = open("names.txt").read().splitlines()

In [ ]:
chars = sorted(list(set("".join(words))))

In [ ]:
chars = ["."] + chars

In [ ]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# Code from Lesson 2

In [ ]:
import torch

In [ ]:
# create the training set of bigrams (x,y)
xs, ys = [], []

for w in words[:]:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):  #zip은 남으면 버린다. ex) .emma.  emma.이기에 2묶음씩 묶어서 앞의 .이 남는데 이것은 버려진다.
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    #print(ch1, ch2)
    xs.append(ix1)
    ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs

tensor([ 0,  5, 13,  ..., 25, 26, 24])

In [ ]:
ys

tensor([ 5, 13, 13,  ..., 26, 24,  0])

# Code from Lesson 3

In [ ]:
# build the dataset

block_size = 1 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:3]:

  print(w)
  context = [0] * block_size
  for ch in w + '.': # ch에는 e가 들어가 있다. w의 첫번째 글자
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
. ---> e
e ---> m
m ---> m
m ---> a
a ---> .
olivia
. ---> o
o ---> l
l ---> i
i ---> v
v ---> i
i ---> a
a ---> .
ava
. ---> a
a ---> v
v ---> a
a ---> .


... -> e
..e -> m
.em -> m
emm -> a
mma -> .

... = 000 -> e =5
..e = 005

In [ ]:
X

tensor([[ 0],
        [ 5],
        [13],
        [13],
        [ 1],
        [ 0],
        [15],
        [12],
        [ 9],
        [22],
        [ 9],
        [ 1],
        [ 0],
        [ 1],
        [22],
        [ 1]])

In [ ]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0])

블럭 사이즈를 1로 하면 lesson 2의 코드가 되고
3으로 하면 lesson 3의 코드가 된다.

# New code

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# 1) names.txt 읽기
# --------------------------------------------------
words = open("names.txt", "r").read().splitlines()

# --------------------------------------------------
# 2) vocabulary 만들기
#    0번은 special token "." 로 고정
# --------------------------------------------------
chars = sorted(list(set("".join(words))))
if "." in chars:
    chars.remove(".")

chars = ["."] + chars

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

vocab_size = len(stoi)
pad_idx = 0   # "." 의 index

In [ ]:
# --------------------------------------------------
# 3) words를 미리 정수 시퀀스로 변환
# --------------------------------------------------
encoded_words = [
    [stoi[ch] for ch in w]
    for w in words
]

In [ ]:
encoded_words

[[5, 13, 13, 1],
 [15, 12, 9, 22, 9, 1],
 [1, 22, 1],
 [9, 19, 1, 2, 5, 12, 12, 1],
 [19, 15, 16, 8, 9, 1],
 [3, 8, 1, 18, 12, 15, 20, 20, 5],
 [13, 9, 1],
 [1, 13, 5, 12, 9, 1],
 [8, 1, 18, 16, 5, 18],
 [5, 22, 5, 12, 25, 14],
 [1, 2, 9, 7, 1, 9, 12],
 [5, 13, 9, 12, 25],
 [5, 12, 9, 26, 1, 2, 5, 20, 8],
 [13, 9, 12, 1],
 [5, 12, 12, 1],
 [1, 22, 5, 18, 25],
 [19, 15, 6, 9, 1],
 [3, 1, 13, 9, 12, 1],
 [1, 18, 9, 1],
 [19, 3, 1, 18, 12, 5, 20, 20],
 [22, 9, 3, 20, 15, 18, 9, 1],
 [13, 1, 4, 9, 19, 15, 14],
 [12, 21, 14, 1],
 [7, 18, 1, 3, 5],
 [3, 8, 12, 15, 5],
 [16, 5, 14, 5, 12, 15, 16, 5],
 [12, 1, 25, 12, 1],
 [18, 9, 12, 5, 25],
 [26, 15, 5, 25],
 [14, 15, 18, 1],
 [12, 9, 12, 25],
 [5, 12, 5, 1, 14, 15, 18],
 [8, 1, 14, 14, 1, 8],
 [12, 9, 12, 12, 9, 1, 14],
 [1, 4, 4, 9, 19, 15, 14],
 [1, 21, 2, 18, 5, 25],
 [5, 12, 12, 9, 5],
 [19, 20, 5, 12, 12, 1],
 [14, 1, 20, 1, 12, 9, 5],
 [26, 15, 5],
 [12, 5, 1, 8],
 [8, 1, 26, 5, 12],
 [22, 9, 15, 12, 5, 20],
 [1, 21, 18, 15, 18, 1],
 

In [ ]:
# --------------------------------------------------
# 4) Dataset 정의
#    각 샘플:
#      x = 길이 block_size의 context
#      y = 다음 문자 1개
# --------------------------------------------------
class NamesContextDataset(Dataset):
    def __init__(self, encoded_words, block_size):
        self.X = []
        self.Y = []
        self.block_size = block_size

        for word in encoded_words:
            context = [0] * block_size   # 왼쪽 padding

            # 마지막 종료 토큰 "." 추가
            for ix in word + [0]:
                self.X.append(context.copy())
                self.Y.append(ix)
                context = context[1:] + [ix]

        self.X = torch.tensor(self.X, dtype=torch.long)
        self.Y = torch.tensor(self.Y, dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


In [ ]:
# --------------------------------------------------
# 5) Bigram용 dataset / dataloader
# --------------------------------------------------
bigram_dataset = NamesContextDataset(
    encoded_words=encoded_words,
    block_size=1
)

bigram_loader = DataLoader(
    bigram_dataset,
    batch_size=32,
    shuffle=True
)

# --------------------------------------------------
# 6) MLP용 dataset / dataloader
# --------------------------------------------------
mlp_dataset = NamesContextDataset(
    encoded_words=encoded_words,
    block_size=3
)

mlp_loader = DataLoader(
    mlp_dataset,
    batch_size=32,
    shuffle=True
)

In [ ]:
for i in bigram_dataset[:10]:
    print(i)

tensor([[ 0],
        [ 5],
        [13],
        [13],
        [ 1],
        [ 0],
        [15],
        [12],
        [ 9],
        [22]])
tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9])


In [ ]:
# --------------------------------------------------
# 7) 확인용 출력 함수
# --------------------------------------------------
def show_samples(dataset, itos, n=5):
    for i in range(n):
        x, y = dataset[i]
        x_chars = [itos[j.item()] for j in x]
        y_char = itos[y.item()]
        print(f"{x_chars} -> {y_char}")

print("=== Bigram samples ===")
show_samples(bigram_dataset, itos, n=5)

print("\n=== MLP samples ===")
show_samples(mlp_dataset, itos, n=5)

# --------------------------------------------------
# 8) batch shape 확인
# --------------------------------------------------
xb_bigram, yb_bigram = next(iter(bigram_loader))
xb_mlp, yb_mlp = next(iter(mlp_loader))

print("\n=== Batch shapes ===")
print("Bigram x:", xb_bigram.shape)  # (32, 1)
print("Bigram y:", yb_bigram.shape)  # (32,)

print("MLP x:", xb_mlp.shape)        # (32, 3)
print("MLP y:", yb_mlp.shape)        # (32,)

=== Bigram samples ===
['.'] -> e
['e'] -> m
['m'] -> m
['m'] -> a
['a'] -> .

=== MLP samples ===
['.', '.', '.'] -> e
['.', '.', 'e'] -> m
['.', 'e', 'm'] -> m
['e', 'm', 'm'] -> a
['m', 'm', 'a'] -> .

=== Batch shapes ===
Bigram x: torch.Size([32, 1])
Bigram y: torch.Size([32])
MLP x: torch.Size([32, 3])
MLP y: torch.Size([32])
